# Machine Unlearning Robustness Pipeline (Kaggle)

This unified notebook allows you to test the robustness of Machine Unlearning under Quantization right here in Kaggle, utilizing the provided GPU accelerators (T4 recommended).


## 1. Environment Initialization
We first need to install the project dependencies, including `bitsandbytes` for 4-bit/8-bit quantization and Hugging Face `accelerate`.

In [1]:
#!pip install -q -U torch transformers datasets bitsandbytes accelerate rouge-score

#!pip uninstall -y torch torchvision torchaudio
#!pip uninstall -y transformers accelerate bitsandbytes

# 2. Install the specific Gemma-3 supported branch
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install git+https://github.com/huggingface/transformers
!pip install --no-cache-dir --upgrade --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
!pip install -q -U datasets bitsandbytes accelerate rouge-score

  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /tmp/pip-req-build-g1u85f07
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-g1u85f07
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Created wheel for transformers: filename=transformers-4.50.0.dev0-py3-none-any.whl size=10936460 sha2

In [2]:
import os
os.chdir(r'/kaggle/working/')

In [3]:
# --- Configuration ---
MODELS_DIR = "models"

# Flags (Set to True to force re-execution even if checkpoints exist)
FLAGS = {
    "RETRAIN_FINE_TUNE": False,
    "RETRAIN_TASK_VECTOR": False,
    "RETRAIN_GRADIENT_ASCENT": False,
    "REQUANTIZE": True 
}

CHECKPOINTS = {
    "FT_FORGET": os.path.join(MODELS_DIR, "fine_tuned_forget"),
    "TV_UNLEARN": os.path.join(MODELS_DIR, "unlearned_task_vector"),
    "GA_UNLEARN": os.path.join(MODELS_DIR, "unlearned_gradient_ascent"),
}

### Hugging Face Login
Gemma requires authentication. Ensure you have added your Hugging Face token as `HF_TOKEN` inside the **Kaggle Secrets** add-on menu.

In [4]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("\n--- Authenticated with Hugging Face successfully! ---")
except Exception as e:
    print("\n--- Warning: Could not find HF_TOKEN in Kaggle Secrets ---")
    print("Please set it up via Add-ons -> Secrets to download Gemma.\n", e)


--- Authenticated with Hugging Face successfully! ---


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from torch.optim import AdamW
from rouge_score import rouge_scorer

MODEL_NAME = "google/gemma-3-1b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 2. Model and Dataset Operations

In [6]:
def load_base_model():
    print(f"Loading Base Model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return model, tokenizer

def load_muse_data():
    """
    Loads the MUSE-Books dataset.
    - 'raw' config for training (text unlearning)
    - 'knowmem' config for evaluation (QA pairs)
    """
    print("Loading MUSE-Books dataset...")
    
    # 1. Load Raw Text (for Unlearning / Fine-tuning)
    # Splits: 'forget', 'retain1' (Retain), 'holdout'
    print("Loading Raw Text (subset='raw')...")
    try:
        raw_dataset = load_dataset("muse-bench/MUSE-Books", "raw")
    except ValueError:
        print("Warning: 'raw' config not found or dataset issue. Fallback to default...")
        raw_dataset = load_dataset("muse-bench/MUSE-Books")

    # Inspect available splits 
    print(f"Available 'raw' splits: {raw_dataset.keys()}")
    forget_train = raw_dataset['forget']
    # retain1 is the standard retention set name in MUSE-Books raw
    retain_train = raw_dataset['retain1'] if 'retain1' in raw_dataset else raw_dataset['retain']
    
    # 2. Load QA Pairs (for Evaluation)
    print("Loading Knowledge Memorization (subset='knowmem')...")
    try:
        qa_dataset = load_dataset("muse-bench/MUSE-Books", "knowmem")
        print(f"Available 'knowmem' splits: {qa_dataset.keys()}")
        
        # Identifying correct QA splits
        # Usually 'forget_qa' or 'forget'
        qa_forget_key = next((k for k in ['forget_qa', 'forget'] if k in qa_dataset), None)
        qa_retain_key = next((k for k in ['retain_qa', 'retain', 'retain1'] if k in qa_dataset), None)

        if qa_forget_key and qa_retain_key:
            forget_qa = qa_dataset[qa_forget_key]
            retain_qa = qa_dataset[qa_retain_key]
        else:
             # Fallback if specific keys aren't found
            print("Warning: Specific QA keys not found. Using available splits.")
            forget_qa = qa_dataset[list(qa_dataset.keys())[0]]
            retain_qa = qa_dataset[list(qa_dataset.keys())[1]] if len(qa_dataset.keys()) > 1 else forget_qa

    except Exception as e:
        print(f"Warning: Could not load knowmem subset ({e}). Using raw text as fallback for evaluation (suboptimal).")
        forget_qa = forget_train
        retain_qa = retain_train

    print(f"Loaded Training Data: {len(forget_train)} forget samples, {len(retain_train)} retain samples.")
    print(f"Loaded Evaluation Data: {len(forget_qa)} forget QA pairs, {len(retain_qa)} retain QA pairs.")
    
    return forget_train, retain_train, forget_qa, retain_qa

## 3. Unlearning Implementations
These functions define Task Arithmetic and Gradient Ascent (Baseline NPO methods).

In [14]:
def compute_task_vector(pretrained_model, finetuned_model):
    task_vector = {}
    # Use the state_dict for easier access
    ft_state = finetuned_model.state_dict()
    
    for name, base_param in pretrained_model.named_parameters():
        if name in ft_state:
            # Move the fine-tuned parameter to the same device as the base parameter
            ft_param = ft_state[name].to(base_param.device)
            task_vector[name] = ft_param - base_param
            
    return task_vector

def apply_task_vector(pretrained_model, task_vector, scaling_factor):
    # Create a copy so we don't modify the original base model in memory
    import copy
    unlearned_model = copy.deepcopy(pretrained_model)
    
    with torch.no_grad():
        for name, param in unlearned_model.named_parameters():
            if name in task_vector:
                # Move the task vector tensor to the same device as this specific layer
                vector_component = task_vector[name].to(param.device)
                
                # Perform the scaled subtraction/addition
                param.add_(vector_component * scaling_factor)
                
    return unlearned_model

def gradient_ascent_unlearning(model, forget_dataloader, epochs=1, lr=1e-5):
    print("Executing standard Gradient Ascent...")
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    # Get the device of the model (handles multi-GPU safely)
    device = next(model.parameters()).device 
    
    for epoch in range(epochs):
        for batch in forget_dataloader:
            # Move all tensors to the model's primary device
            inputs = batch['input_ids'].to(device)
            masks = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=inputs, attention_mask=masks, labels=labels)
            
            # Gradient ascent: we want to MAXIMIZE loss (negative loss)
            loss = -outputs.loss 
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
    return model

In [15]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

def tokenize_dataset(dataset, tokenizer, max_length=512):
    """
    Tokenizes the dataset for training/unlearning. 
    Assumes a text column exists.
    """
    # Identify the text column
    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    
    def tokenize_function(examples):
        return tokenizer(examples[text_col], padding="max_length", truncation=True, max_length=max_length)
    
    # Check if already tokenized
    if "input_ids" in dataset.column_names:
        return dataset
        
    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
    tokenized_dataset.set_format("torch")
    return tokenized_dataset

def create_dataloader(dataset, tokenizer, batch_size=4):
    tokenized = tokenize_dataset(dataset, tokenizer)
    # The collator automatically creates 'labels' from 'input_ids'
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    return DataLoader(
        tokenized, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=data_collator # Add this line
    )

def fine_tune_model(model, tokenizer, dataset, output_dir, epochs=3):
    """
    Fine-tunes the model on the provided dataset (usually the forget set for Task Vector).
    """
    print(f"Fine-tuning model on {len(dataset)} samples...")
    
    # Tokenize
    tokenized_dataset = tokenize_dataset(dataset, tokenizer)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2, # Small batch size for T4
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no", 
        logging_steps=10,
        fp16=False,
        bf16=False,
        report_to="none",
        optim="paged_adamw_8bit" # Efficient optimizer
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    
    trainer.train()
    return model

def prepare_cloze_questions(dataset):
    """
    Attempts to prepare cloze questions from the dataset for evaluation.
    Expects 'question' and 'answer' columns, or similar.
    """
    questions = []
    # Check for common QA column names
    q_col = next((col for col in ['question', 'prompt', 'input'] if col in dataset.column_names), None)
    a_col = next((col for col in ['answer', 'target', 'output'] if col in dataset.column_names), None)
    
    if q_col and a_col:
        for item in dataset:
            questions.append({
                'prompt': item[q_col],
                'answer': item[a_col]
            })
    else:
        # Fallback: create dummy tasks or just warn
        print("Warning: Could not identify question/answer columns for Cloze Evaluation. Evaluation might be skipped or inaccurate.")
        # If dataset is text only, maybe treating start as prompt?
        # For valid unlearning evaluation, we usually need specific queries. 
        # Using first 50 chars as prompt and next 10 as answer (naive)
        if 'text' in dataset.column_names:
            for item in dataset.select(range(min(50, len(dataset)))):
                text = item['text']
                if len(text) > 100:
                    questions.append({
                        'prompt': text[:50],
                        'answer': text[50:60] # Naive
                    })
    
    return questions

## 4. Post-Training Quantization (PTQ)
Loading the unlearned FP16 models in 4-bit to observe the Recovery Delta.

In [9]:
def load_quantized_model(model_dir_path, bit_width=4):
    print(f"Quantizing model from {model_dir_path} to {bit_width}-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(bit_width == 4),
        load_in_8bit=(bit_width == 8),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_dir_path,
        device_map="auto",
        quantization_config=bnb_config
    )
    return quantized_model

## 5. Evaluation Loop

In [10]:
def evaluate_factual_recall(model, tokenizer, cloze_questions):
    correct = 0
    for question_dict in cloze_questions:
        prompt = question_dict['prompt']
        target_answer = question_dict['answer'].lower()
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=10)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        continuation = generated_text[len(prompt):].lower()
        if target_answer in continuation:
            correct += 1
            
    return correct / len(cloze_questions) if cloze_questions else 0

## Execution Sandbox
Run your components below to generate the metrics outlined in your proposal.

In [16]:
import os
import shutil
import torch
import pandas as pd
from transformers import AutoModelForCausalLM
import gc

os.makedirs(MODELS_DIR, exist_ok=True)

# --- Execution Pipeline ---

print("\n=== PHASE 1: Loading Resources ===")
# Load Data
forget_train, retain_train, forget_qa, retain_qa = load_muse_data()

# Prepare Evaluation Data
# Generate questions from the QA subset (first 30 examples for speed)
# Using separate QA pairs ensures our 'answer' is ground truth, not raw text slicing
forget_questions = prepare_cloze_questions(forget_qa.select(range(min(30, len(forget_qa)))))
retain_questions = prepare_cloze_questions(retain_qa.select(range(min(30, len(retain_qa)))))
print(f"Prepared {len(forget_questions)} forget questions and {len(retain_questions)} retain questions.")

# Load Tokenizer (Base model needed momentarily for tokenizer if not separate)
print("Loading Base Model Tokenizer...")
_, tokenizer = load_base_model()
del _ # Unload model if loaded, keep tokenizer
torch.cuda.empty_cache()
gc.collect()

# --- PHASE 2: Fine-Tuning (Task Vector Prep) ---
print("\n=== PHASE 2: Fine-Tuning (Task Vector Prep) ===")
if not os.path.exists(CHECKPOINTS["FT_FORGET"]) or FLAGS["RETRAIN_FINE_TUNE"]:
    print("Starting Fine-Tuning on Forget Set (Raw Text)...")
    base_model, _ = load_base_model() # Load fresh
    ft_model = fine_tune_model(base_model, tokenizer, forget_train, CHECKPOINTS["FT_FORGET"], epochs=3)
    print(f"Saving Fine-Tuned Model to {CHECKPOINTS['FT_FORGET']}...")
    ft_model.save_pretrained(CHECKPOINTS["FT_FORGET"])
    tokenizer.save_pretrained(CHECKPOINTS["FT_FORGET"])
    
    del base_model, ft_model
    torch.cuda.empty_cache()
    gc.collect()
else:
    print(f"Checkpoint found: {CHECKPOINTS['FT_FORGET']}")

# --- PHASE 3: Task Vector Unlearning ---
print("\n=== PHASE 3: Task Vector Unlearning ===")
if not os.path.exists(CHECKPOINTS["TV_UNLEARN"]) or FLAGS["RETRAIN_TASK_VECTOR"]:
    print("Computing Task Vector...")
    base_model, _ = load_base_model()
    ft_model = AutoModelForCausalLM.from_pretrained(CHECKPOINTS["FT_FORGET"], torch_dtype=torch.float16, device_map="auto")
    
    task_vector = compute_task_vector(base_model, ft_model)
    # Scaling factor logic: (Base + (FineTuned - Base) * -scaling) -> Unlearn
    scaling_factor = -0.5 
    tv_unlearned_model = apply_task_vector(base_model, task_vector, scaling_factor=scaling_factor)
    
    print(f"Saving TV Unlearned Model to {CHECKPOINTS['TV_UNLEARN']}...")
    tv_unlearned_model.save_pretrained(CHECKPOINTS["TV_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["TV_UNLEARN"])
    
    del base_model, ft_model, tv_unlearned_model, task_vector
    torch.cuda.empty_cache()
    gc.collect()
else:
    print(f"Checkpoint found: {CHECKPOINTS['TV_UNLEARN']}")

# --- PHASE 4: Gradient Ascent Unlearning ---
print("\n=== PHASE 4: Gradient Ascent Unlearning ===")
if not os.path.exists(CHECKPOINTS["GA_UNLEARN"]) or FLAGS["RETRAIN_GRADIENT_ASCENT"]:
    print("Executing Gradient Ascent...")
    base_model, _ = load_base_model()
    # Create DataLoader just for Forget Set (Raw Text)
    forget_loader = create_dataloader(forget_train, tokenizer, batch_size=1)
    
    ga_unlearned_model = gradient_ascent_unlearning(base_model, forget_loader, epochs=1, lr=5e-5)
    
    print(f"Saving GA Unlearned Model to {CHECKPOINTS['GA_UNLEARN']}...")
    ga_unlearned_model.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
    
    del base_model, ga_unlearned_model
    torch.cuda.empty_cache()
    gc.collect()
else:
    print(f"Checkpoint found: {CHECKPOINTS['GA_UNLEARN']}")

# --- PHASE 5: Evaluation & Quantization ---
print("\n=== PHASE 5: Evaluation & Quantization ===")
results = []

def run_evaluation(model_obj_or_path, model_name, is_quantized=False):
    print(f"Evaluating {model_name}...")
    try:
        if isinstance(model_obj_or_path, str):
            if is_quantized:
                model = load_quantized_model(model_obj_or_path, bit_width=4)
            else:
                model = AutoModelForCausalLM.from_pretrained(model_obj_or_path, torch_dtype=torch.float16, device_map="auto")
        else:
            model = model_obj_or_path

        forget_score = evaluate_factual_recall(model, tokenizer, forget_questions)
        retain_score = evaluate_factual_recall(model, tokenizer, retain_questions)
        
        # Helper cleanup
        if isinstance(model_obj_or_path, str):
            del model
            torch.cuda.empty_cache()
            gc.collect()
            
        return {
            "Model": model_name,
            "Forget Recall": forget_score,
            "Retain Recall": retain_score
        }
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        return {"Model": model_name, "Forget Recall": "Error", "Retain Recall": "Error"}

# 1. Base Model
print("Evaluating Base Model...")
base_model, _ = load_base_model()
results.append(run_evaluation(base_model, "Base Model (FP16)"))
del base_model
torch.cuda.empty_cache()
gc.collect()

# 2. Unlearned Models (FP16 & 4-bit)
checkpoints = [
    ("TV Unlearned", CHECKPOINTS["TV_UNLEARN"]),
    ("GA Unlearned", CHECKPOINTS["GA_UNLEARN"])
]

for name, path in checkpoints:
    # FP16 Evaluation
    results.append(run_evaluation(path, f"{name} (FP16)", is_quantized=False))
    # 4-bit Quantization & Evaluation
    if FLAGS["REQUANTIZE"]:
        results.append(run_evaluation(path, f"{name} (4-bit)", is_quantized=True))

# --- Summary ---
print("\n=== Final Results ===")
df_results = pd.DataFrame(results)
print(df_results)
df_results.to_csv("unlearning_benchmark_results.csv", index=False)
print("Pipeline Successfully Completed!")


=== PHASE 1: Loading Resources ===
Loading MUSE-Books dataset...
Loading Raw Text (subset='raw')...
Available 'raw' splits: dict_keys(['retain2', 'forget', 'retain1', 'holdout'])
Loading Knowledge Memorization (subset='knowmem')...
Available 'knowmem' splits: dict_keys(['retain_qa_icl', 'retain_qa', 'forget_qa', 'forget_qa_icl'])
Loaded Training Data: 4 forget samples, 12 retain samples.
Loaded Evaluation Data: 100 forget QA pairs, 100 retain QA pairs.
Prepared 30 forget questions and 30 retain questions.
Loading Base Model Tokenizer...
Loading Base Model google/gemma-3-1b-it...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


=== PHASE 2: Fine-Tuning (Task Vector Prep) ===
Checkpoint found: models/fine_tuned_forget

=== PHASE 3: Task Vector Unlearning ===
Checkpoint found: models/unlearned_task_vector

=== PHASE 4: Gradient Ascent Unlearning ===
Executing Gradient Ascent...
Loading Base Model google/gemma-3-1b-it...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Executing standard Gradient Ascent...
Saving GA Unlearned Model to models/unlearned_gradient_ascent...


...

/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:112: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


Error evaluating GA Unlearned (FP16): CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Evaluating GA Unlearned (4-bit)...
Quantizing model from models/unlearned_gradient_ascent to 4-bit precision...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3ForCausalLM LOAD REPORT from: models/unlearned_gradient_ascent
Key                                           | Status     |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

Error evaluating GA Unlearned (4-bit): We encountered some issues during automatic conversion of the weights. For details look at the `CONVERSION` entries of the above report!

=== Final Results ===
                  Model Forget Recall Retain Recall
0     Base Model (FP16)      0.066667      0.133333
1   TV Unlearned (FP16)      0.033333           0.1
2  TV Unlearned (4-bit)      0.066667           0.1
3   GA Unlearned (FP16)         Error         Error
4  GA Unlearned (4-bit)         Error         Error
Pipeline Successfully Completed!
